# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q emoji wordcloud textblob
    print('Packages installed.')


# WhatsApp Chat Analysis In-Depth

This notebook provides a comprehensive toolkit for analyzing exported WhatsApp chat data. It goes beyond simple counting to include sentiment analysis over time, response time analysis, and activity heatmaps.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from datetime import datetime, timedelta
import emoji
from collections import Counter
from textblob import TextBlob
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style("whitegrid")


## 1. Data Loading and Robust Parsing

We use a regex-based parser that handles both Android and iOS formats, including multi-line messages.

In [ ]:
def parse_whatsapp_chat(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        data = f.readlines()
        
    messages = []
    # Android: 12/04/2023, 14:30 - User: Message
    android_regex = r'^(\d{1,2}/\d{1,2}/\d{2,4}), (\d{1,2}:\d{2}) - (.*?): (.*)$'
    # iOS: [12/04/2023, 14:30:15] User: Message
    ios_regex = r'^\[(\d{1,2}/\d{1,2}/\d{2,4}), (\d{1,2}:\d{2}:\d{2})\] (.*?): (.*)$'
    
    for line in data:
        line = line.strip()
        if not line:
            continue
            
        # Try Android format
        match = re.match(android_regex, line)
        if match:
            date, time, sender, message = match.groups()
            messages.append([date, time, sender, message])
            continue
            
        # Try iOS format
        match = re.match(ios_regex, line)
        if match:
            date, time, sender, message = match.groups()
            messages.append([date, time, sender, message])
            continue
            
        # Handle multi-line messages
        if messages:
            messages[-1][-1] += " " + line

    df = pd.DataFrame(messages, columns=['Date', 'Time', 'Sender', 'Message'])
    
    # Convert Date and Time to datetime objects
    # Note: We try multiple formats to be robust
    try:
        df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M')
    except ValueError:
        try:
             df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%y %H:%M:%S')
        except ValueError:
             # Try mixed US/EU formats if necessary, or just ignore failed rows
             pass
    
    return df

# Create a more complex dummy dataset
dummy_data = [
    "12/01/2023, 10:00 - Alice: Hey everyone! How are we doing?",
    "12/01/2023, 10:05 - Bob: Good morning Alice! I am doing great.",
    "12/01/2023, 10:06 - Charlie: Hi guys. I'm running a bit late.",
    "12/01/2023, 10:10 - Alice: No worries Charlie. We haven't started yet.",
    "13/01/2023, 09:00 - Bob: Did anyone see the news?",
    "13/01/2023, 09:15 - Alice: Yes, it is crazy! 😱",
    "13/01/2023, 09:20 - Charlie: I missed it. What happened?",
    "14/01/2023, 18:00 - Alice: Meeting at 5pm tomorrow?",
    "14/01/2023, 18:05 - Bob: Works for me.",
    "14/01/2023, 18:10 - Charlie: 👍",
    "15/01/2023, 22:00 - Alice: I am so tired.",
    "15/01/2023, 22:01 - Bob: Go to sleep!",
    "16/01/2023, 08:00 - Charlie: Wakey wakey!"
]

with open('dummy_chat.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(dummy_data))

df = parse_whatsapp_chat('dummy_chat.txt')
print(df.head())

## 2. Response Time Analysis
We calculate the time difference between consecutive messages to see who replies the fastest.

In [ ]:
if 'datetime' in df.columns:
    df['response_time'] = df['datetime'].diff().dt.total_seconds() / 60  # in minutes
    
    # Filter out long gaps (e.g., > 60 mins) which likely indicate new conversations
    conv_df = df[df['response_time'] < 60]
    
    avg_response = conv_df.groupby('Sender')['response_time'].mean()
    
    plt.figure(figsize=(8, 5))
    avg_response.plot(kind='bar', color='salmon')
    plt.title('Average Response Time (Minutes)')
    plt.ylabel('Minutes')
    plt.show()

## 3. Sentiment Over Time
Using TextBlob to track the emotional tone of the conversation.

In [ ]:
def get_sentiment(text):
    return TextBlob(text).sentiment.polarity

df['sentiment'] = df['Message'].apply(get_sentiment)

plt.figure(figsize=(12, 6))
plt.plot(df['datetime'], df['sentiment'], marker='o', linestyle='-')
plt.axhline(0, color='gray', linestyle='--')
plt.title('Sentiment Over Time')
plt.xlabel('Date')
plt.ylabel('Sentiment (Polarity)')
plt.show()

## 4. Activity Heatmap
Visualizing when the group is most active (Day of Week vs Hour of Day).

In [ ]:
if 'datetime' in df.columns:
    df['DayOfWeek'] = df['datetime'].dt.day_name()
    df['Hour'] = df['datetime'].dt.hour
    
    # Create pivot table
    heatmap_data = df.pivot_table(index='DayOfWeek', columns='Hour', values='Message', aggfunc='count', fill_value=0)
    
    # Sort days properly
    days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    heatmap_data = heatmap_data.reindex(days_order)
    
    plt.figure(figsize=(12, 6))
    sns.heatmap(heatmap_data, cmap='YlGnBu', annot=True, fmt='g')
    plt.title('Chat Activity Heatmap')
    plt.show()

## 5. Word Cloud
Visualizing the most common words.

In [ ]:
from wordcloud import WordCloud

text = " ".join(message for message in df.Message)
text_cleaned = re.sub(r'http\S+', '', text) # remove urls
text_cleaned = re.sub(r'[^a-zA-Z\s]', '', text_cleaned) # remove special chars

if text_cleaned.strip():
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text_cleaned)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud')
    plt.show()
else:
    print("Not enough text for WordCloud")